# (Appendix) A2A Protocol Multi-Agent (Agent-to-Agent Multi-Agent)
## Splitting work among multiple agents that communicate via `a2a-sdk`

> 📦 **Environment installation and run commands** are summarized in [`env_guides/M02_5_a2a_multiagent_eng.md`](env_guides/M02_5_a2a_multiagent_eng.md).
> This notebook extends the A2A concepts covered in [`M02_3_mcp_a2a_eng.ipynb`](M02_3_mcp_a2a_eng.ipynb)
> using the actual reference implementation, **`a2a-sdk` (a2a-python) 1.1.0**.

---

### What is the A2A protocol?
**A2A (Agent-to-Agent)** is an open protocol published by Google that enables agents built with different frameworks and vendors
to **interoperate**. Its core concepts are as follows.

- **AgentCard** — the agent's "business card". It contains the name, description, and version, the **skills (AgentSkill)** it provides,
  its **capabilities (AgentCapabilities, e.g. streaming)**, and its connection endpoints (**AgentInterface**: URL + transport binding).
  It is **discovered** at the `/.well-known/agent-card.json` path.
- **Message / Part** — messages exchanged between agents. Composed of multiple **Parts** such as text, data, and files.
- **Task** — a single unit of work. It has a state (`submitted → working → completed/failed`) and result **Artifacts**.
- **Transport binding** — JSON-RPC / gRPC / HTTP+JSON. The client reads the card and connects via a supported binding.

### What this notebook demonstrates
1. Define three specialist agents (calculation, summarization, writing) with **AgentCard/AgentSkill**
2. Launch each agent as a **real A2A HTTP server** (in a background thread)
3. **Discover cards** with `A2ACardResolver` and create **A2A clients** with `ClientFactory`
4. A **coordinator** uses the LLM to **decompose** a compound request, **delegates via A2A messages** to each specialist agent,
   and then **aggregates the results** with the writer agent

### Prerequisites
- Windows 11 + **CMD (`cmd.exe`)** + Python **3.11** (managed by `uv`), kernel = **`Agentic AI (uv)`**
- Default LLM = **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`**. This LLM is used as the brain of each specialist agent.
- Requires `a2a-sdk` / `uvicorn` / `httpx` (installed automatically by the setup cell below).

### Architecture overview
```
[User compound request]
        │
   [Coordinator] ── decomposes into subtasks with the LLM
        │  A2A messages (JSON-RPC over HTTP)
        ├──▶ [math_agent  : A2A HTTP server]  (LLM API)
        ├──▶ [research_agent: A2A HTTP server] (LLM API)
        └──▶ [writer_agent  : A2A HTTP server] (LLM API) ── aggregates results
```


In [1]:
# [setup] Self-contained setup — lets this notebook run on its own.
import sys, os
sys.path.insert(0, os.path.abspath(''))   # add notebooks/ to the import path

import utils
utils.reload_env()   # reload .env (refresh LLM_PROVIDER, etc.)

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text   # provider-agnostic response normalization (including <think> removal)

# Packages needed for the A2A exercise (A2A standard SDK + ASGI server + async HTTP client)
utils.uv_install(['a2a-sdk', 'uvicorn', 'httpx',
                  'langchain', 'langchain-openai'])

llm = utils.get_llm()   # LLM used as the brain of each specialist agent (default nvidia/deepseek-v4.1-flash)
print("Ready:", utils.LLM_PROVIDER)


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct
[uv] 설치 완료: ['a2a-sdk', 'uvicorn', 'httpx', 'langchain', 'langchain-openai']


준비 완료: nvidia


---
## 1. A2A SDK Imports and Common Helpers

We use the actual symbols of `a2a-sdk` 1.1.0 (the API differs significantly between versions, so this follows the installed version).

| Symbol | Role |
|---|---|
| `AgentCard`, `AgentSkill`, `AgentCapabilities`, `AgentInterface` | Agent specification (for discovery) |
| `Message`, `Part`, `Role`, `Task`, `TaskState` | Message and task types (protocol-buffer based) |
| `AgentExecutor`, `RequestContext`, `EventQueue` | **Server-side** agent execution model |
| `TaskUpdater` | Helper that writes task status and results (Artifacts) to the event queue |
| `DefaultRequestHandler`, `InMemoryTaskStore` | Request handler + task store |
| `create_agent_card_routes`, `create_jsonrpc_routes` | Create Starlette routes (→ served with `uvicorn`) |
| `ClientFactory`, `ClientConfig`, `A2ACardResolver` | **Client-side** card discovery + connection |
| `new_text_message`, `new_text_part`, `new_task`, `get_stream_response_text` | Message/task creation and parsing helpers |

The cell below defines the imports and small helpers reused later (`free_port`, `make_agent_card`, `LLMAgentExecutor`).
`LLMAgentExecutor` is the **agent body** that the A2A server calls for every incoming request; it drives the LLM with a role-specific system prompt.


In [2]:
import asyncio, threading, time, socket, json, re
import httpx, uvicorn
from starlette.applications import Starlette
from langchain_core.messages import SystemMessage, HumanMessage

# --- actual a2a-sdk 1.1.0 symbols ---
from a2a.types import (
    AgentCard, AgentSkill, AgentCapabilities, AgentInterface,
    Role, SendMessageRequest, TaskState,
)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.client import ClientFactory, ClientConfig, A2ACardResolver
from a2a.utils import TransportProtocol
from a2a.helpers.proto_helpers import (
    new_text_message, new_text_part, new_task, get_stream_response_text,
)


def free_port() -> int:
    '''Returns an arbitrary available local port (avoids port conflicts).'''
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port


class LLMAgentExecutor(AgentExecutor):
    '''Server-side A2A executor that drives the LLM with a role-specific system prompt.

    The A2A server calls ``execute(context, event_queue)`` for each request.
    Here we extract the user input, pass it to the LLM, record the result in the
    Task's Artifact (output) via ``TaskUpdater``, and then mark it as completed.
    '''

    def __init__(self, llm, system_prompt: str):
        self.llm = llm                      # this agent's brain (LangChain model)
        self.system_prompt = system_prompt  # role definition (calculation/summarization/writing, etc.)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_text = context.get_user_input()          # text of the incoming Message
        # For a new request, the Task must be enqueued first (must precede status events)
        if context.current_task is None:
            await event_queue.enqueue_event(
                new_task(context.task_id, context.context_id, TaskState.TASK_STATE_SUBMITTED)
            )
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.start_work()                     # state: working
        # The LLM call is synchronous, so offload it to a thread to avoid blocking the event loop
        answer = await asyncio.to_thread(
            bootstrap.invoke_text, self.llm,
            [SystemMessage(self.system_prompt), HumanMessage(user_text)],
        )
        await updater.add_artifact([new_text_part(answer)], name="result")  # attach the result
        await updater.complete(                          # state: completed
            message=updater.new_agent_message([new_text_part(answer)])
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        '''Cancellation is not supported in this example.'''
        raise NotImplementedError("cancel is not supported in this demo")


print("A2A symbols imported. TransportProtocol.JSONRPC =", TransportProtocol.JSONRPC.value)


A2A 심볼 임포트 완료. TransportProtocol.JSONRPC = JSONRPC


---
## 2. Defining the Specialist Agents (AgentCard · AgentSkill)

Each agent publishes an **AgentCard** describing itself. The card contains the **skills (AgentSkill)** it provides,
its **capabilities (AgentCapabilities)**, and its connection **interfaces (AgentInterface: URL + transport binding)**.
Clients/coordinators **discover** this card to learn what the agent can do and then connect to it.

Here we set up three specialist agents.

| Agent | Role (system prompt) | Skill |
|---|---|---|
| `math_agent` | Accurate arithmetic calculation | `calc` |
| `research_agent` | Briefly summarizes and explains a topic | `summarize` |
| `writer_agent` | Organizes material into a single paragraph | `compose` |

`make_agent_card()` builds a card with a JSON-RPC interface bound to the given port.


In [3]:
def make_agent_card(name, description, skill_id, skill_name, port) -> AgentCard:
    '''Creates an AgentCard with a JSON-RPC interface for a specialist agent.'''
    return AgentCard(
        name=name,
        description=description,
        version="1.0.0",
        # Endpoint + transport binding (JSON-RPC). Clients connect using this information.
        supported_interfaces=[AgentInterface(
            url=f"http://127.0.0.1:{port}/",
            protocol_binding=TransportProtocol.JSONRPC.value,
            protocol_version="1.0",
        )],
        capabilities=AgentCapabilities(streaming=True),   # capability: streaming supported
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[AgentSkill(id=skill_id, name=skill_name,
                           description=description, tags=[skill_id])],
    )


# (role prompt, skill id, skill name) — the key is the agent's identifier in A2A
AGENT_SPECS = {
    "math_agent":     ("You are a calculation specialist agent. Perform only arithmetic calculations accurately and answer concisely with just the final number.",
                       "calc", "Calculation"),
    "research_agent": ("You are a research specialist agent. Summarize and explain the given topic simply in two sentences or fewer.",
                       "summarize", "Summarization"),
    "writer_agent":   ("You are a writing specialist agent. Organize the material you receive as input into a single natural paragraph.",
                       "compose", "Writing"),
}

# Create cards + executors (LLM brains)
agent_cards = {}
agent_executors = {}
for _name, (_prompt, _sid, _sname) in AGENT_SPECS.items():
    _port = free_port()
    agent_cards[_name] = make_agent_card(_name, _prompt, _sid, _sname, _port)
    agent_executors[_name] = LLMAgentExecutor(llm, _prompt)

for _name, _card in agent_cards.items():
    print(f"{_name:15s} -> {_card.supported_interfaces[0].url}  skills={[s.name for s in _card.skills]}")


math_agent      -> http://127.0.0.1:51789/  skills=['계산']
research_agent  -> http://127.0.0.1:51790/  skills=['요약']
writer_agent    -> http://127.0.0.1:51791/  skills=['작문']


---
## 3. Launching Each Agent as a Real A2A HTTP Server

An A2A server consists of the following.

1. `DefaultRequestHandler(agent_executor, task_store, agent_card)` — passes incoming requests to the executor and stores tasks
2. `create_agent_card_routes(card)` + `create_jsonrpc_routes(handler, rpc_url="/")` — Starlette routes
3. Serve the `Starlette(routes=...)` app with `uvicorn`

To avoid blocking notebook cells, each server runs in a **background thread** (in production, each agent runs as an
independent service/container). We wait for readiness via `server.started` before moving to the next cell.


In [4]:
def start_a2a_server(card: AgentCard, executor: AgentExecutor):
    '''Starts an A2A HTTP server for the AgentCard/executor in a background thread and returns (server, thread).'''
    handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),   # in-memory task store for the demo
        agent_card=card,
    )
    # Card discovery route (/.well-known/agent-card.json) + JSON-RPC route (/)
    routes = create_agent_card_routes(card) + create_jsonrpc_routes(handler, rpc_url="/")
    app = Starlette(routes=routes)

    port = int(card.supported_interfaces[0].url.rstrip("/").split(":")[-1])
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)  # run as a daemon thread
    thread.start()
    for _ in range(50):          # wait up to ~5 seconds for the server to open its socket
        if server.started:
            break
        time.sleep(0.1)
    return server, thread


# Start the three agent servers
running_servers = {}
for _name in agent_cards:
    _srv, _th = start_a2a_server(agent_cards[_name], agent_executors[_name])
    running_servers[_name] = (_srv, _th)
    print(f"{_name:15s} started={_srv.started}  {agent_cards[_name].supported_interfaces[0].url}")


math_agent      started=True  http://127.0.0.1:51789/


research_agent  started=True  http://127.0.0.1:51790/


writer_agent    started=True  http://127.0.0.1:51791/


---
## 4. A2A Card Discovery

If a client knows only an agent's URL, it can use `A2ACardResolver` to read `/.well-known/agent-card.json`
and fetch that agent's **AgentCard**. This is A2A **discovery**.
Below, we fetch the cards over real HTTP and check their names, skills, and transport bindings.

> This notebook uses the async A2A API, so cells use **top-level `await`** (supported by Jupyter).


In [5]:
# A single async HTTP client reused across cells
http_client = httpx.AsyncClient(timeout=120)

# Assume we know only each agent's URL, and discover the cards one by one.
# In practice these may be different hosts/services, so discovery is repeated from just the list of URLs.
discovered_cards = {}
for _name, _card in agent_cards.items():
    _url = _card.supported_interfaces[0].url.rstrip("/")
    resolver = A2ACardResolver(http_client, base_url=_url)
    discovered = await resolver.get_agent_card()   # discover /.well-known/agent-card.json
    discovered_cards[_name] = discovered

# Print information for all discovered cards
for _name, discovered in discovered_cards.items():
    print("=" * 60)
    print("Discovered card name :", discovered.name)
    print("Description          :", discovered.description)
    print("Skills               :", [(s.id, s.name) for s in discovered.skills])
    print("Transport bindings   :", [i.protocol_binding for i in discovered.supported_interfaces])
    print("Streaming supported  :", discovered.capabilities.streaming)
print("=" * 60)
print(f"Discovered {len(discovered_cards)} agent cards in total")


발견한 카드 이름 : math_agent
설명            : 너는 계산 전문 에이전트다. 산술 계산만 정확히 수행하고 최종 숫자만 간결히 답한다.
스킬            : [('calc', '계산')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
발견한 카드 이름 : research_agent
설명            : 너는 조사 전문 에이전트다. 주어진 주제를 2문장 이내로 쉽게 요약 설명한다.
스킬            : [('summarize', '요약')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
발견한 카드 이름 : writer_agent
설명            : 너는 작문 전문 에이전트다. 입력으로 받은 자료들을 자연스러운 한 단락으로 정리한다.
스킬            : [('compose', '작문')]
전송 바인딩      : ['JSONRPC']
스트리밍 지원    : True
총 3개 에이전트 카드 발견 완료


---
## 5. Calling a Single Agent with an A2A Client

Create a client matching the card with `ClientFactory(ClientConfig(...))`, and send a `Message` inside a
`SendMessageRequest`. The response arrives as a `StreamResponse` stream (here, with `streaming=False`, it arrives as
a completed Task), and the text is extracted with `get_stream_response_text()`.

`ask_agent()` is a helper that the coordinator later reuses to **delegate work** to each specialist agent.


In [6]:
async def ask_agent(card: AgentCard, text: str) -> str:
    '''Sends an A2A message to the agent of the given AgentCard and returns the response text.'''
    config = ClientConfig(
        httpx_client=http_client,
        streaming=False,                                        # receive as a completed Task
        supported_protocol_bindings=[TransportProtocol.JSONRPC.value],
    )
    client = ClientFactory(config).create(card)                 # card → A2A client
    request = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER))
    parts = []
    async for response in client.send_message(request):         # consume the response stream
        chunk = get_stream_response_text(response)
        if chunk:
            parts.append(chunk)
    return "\n".join(parts)


# Single-agent call demo: send an A2A message to the calculation agent
_answer = await ask_agent(agent_cards["math_agent"], "What is 3200 times 4?")
print("math_agent response :", _answer)


math_agent 응답 : 12800


---
## 6. Coordinator: Decompose Compound Request → Delegate via A2A → Aggregate

Now we build a **coordinator** so that multiple agents **collaborate**. The coordinator

1. uses the LLM to **decompose** the user's **compound request** into **subtasks** (specifying which specialist agent handles each),
2. **delegates** each subtask to the corresponding specialist agent **via A2A messages** (`ask_agent`), and
3. passes the collected results to the **writer agent (`writer_agent`)** to **aggregate them into a single answer**.

If the decomposition result (JSON) is malformed, it safely **falls back** (sends the original request to all specialist agents).


In [7]:
class Coordinator:
    '''Coordinator that decomposes compound requests, delegates them to specialist agents via A2A, and aggregates the results.'''

    def __init__(self, llm, cards: dict):
        self.llm = llm
        self.cards = cards
        # writer_agent handles the final aggregation, so it is excluded from the decomposition targets (workers)
        self.workers = {n: c for n, c in cards.items() if n != "writer_agent"}

    async def decompose(self, request: str) -> list:
        '''Uses the LLM to decompose the request into [{agent, subtask}, ...] (robust parsing + fallback).'''
        roster = "\n".join(f"- {n}: {c.description}" for n, c in self.workers.items())
        sys_prompt = (
            "You are a coordinator. Decompose the user request into subtasks to be handled by the specialist agents below.\n"
            + roster
            + "\nOutput only a JSON array. Each element is {\"agent\": agent_name, \"subtask\": instruction}."
            " Do not write any other explanation."
        )
        raw = await asyncio.to_thread(
            bootstrap.invoke_text, self.llm,
            [SystemMessage(sys_prompt), HumanMessage(request)],
        )
        try:
            plan = json.loads(re.search(r"\[.*\]", raw, re.S).group(0))
            plan = [s for s in plan if s.get("agent") in self.workers and s.get("subtask")]
            if plan:
                return plan
        except Exception:
            pass
        # Fallback: if format parsing fails, delegate the original request as is to all specialist agents
        return [{"agent": n, "subtask": request} for n in self.workers]

    async def run(self, request: str) -> str:
        '''Decompose request → delegate to each specialist agent via A2A → aggregate with writer_agent.'''
        plan = await self.decompose(request)
        print("[Decomposed plan]")
        for step in plan:
            print(f"  - {step['agent']}: {step['subtask']}")

        results = []
        for step in plan:                                  # delegate each subtask via A2A
            answer = await ask_agent(self.cards[step["agent"]], step["subtask"])
            results.append((step["agent"], answer))
            print(f"\n[{step['agent']} response] {answer}")

        material = "\n".join(f"- {name}: {ans}" for name, ans in results)
        final = await ask_agent(                            # aggregate results (writer agent)
            self.cards["writer_agent"],
            f"Based on the material below, write the final answer to the user's question as a single paragraph.\n"
            f"Question: {request}\nMaterial:\n{material}",
        )
        return final


coordinator = Coordinator(llm, agent_cards)
user_request = "Calculate the total price of 4 cups of coffee at 3,200 won each, and explain what espresso is."
final_answer = await coordinator.run(user_request)
print("\n" + "=" * 60)
print("[Coordinator final answer]")
print(final_answer)


[분해된 계획]
  - math_agent: 3200 * 4 계산
  - research_agent: 에스프레소 설명



[math_agent 응답] 12800



[research_agent 응답] 에스프레소는 이탈리아에서 유래한 커피의 한 종류로, 강한 향과 풍부한 맛을 특징으로 합니다. 에스프레소는 고온의 물을 고압으로 강제로 커피콩에 통과시켜 만든 커피로, 일반적으로 1-2ml의 양으로 추출됩니다.



[코디네이터 최종 답]
3200원짜리 커피 4잔의 총액을 계산하면 3200 x 4 = 12800원입니다. 에스프레소는 이탈리아에서 유래한 커피의 한 종류로, 강한 향과 풍부한 맛을 특징으로 하는 커피로, 일반적으로 1-2ml의 양으로 추출됩니다.


---
## 7. Cleanup and Server Shutdown

When the exercise is done, close the HTTP client and shut down the background A2A servers.


In [8]:
# Close the async HTTP client
await http_client.aclose()

# Request shutdown of the background A2A servers (daemon threads, so they are also cleaned up when the kernel exits)
for _name, (_srv, _th) in running_servers.items():
    _srv.should_exit = True
print("Client/server cleanup complete")


클라이언트/서버 정리 완료


---
## 8. Wrap-up — A2A Essentials and Production Deployment

### A2A elements actually used in this notebook
- **AgentCard / AgentSkill / AgentCapabilities / AgentInterface** — make agents self-describing and **discoverable**
- **A2ACardResolver** — discovers `/.well-known/agent-card.json`
- **AgentExecutor / RequestContext / EventQueue / TaskUpdater** — server-side execution model (request → Task → Artifact)
- **Message / Task / Artifact** — protocol types for requests, tasks, and results
- **ClientFactory / ClientConfig** — card-based client creation, delegation via `SendMessageRequest`

### Production deployment
- In the notebook, each agent was launched in a **background thread** for convenience, but in practice each agent runs as an
  **independent service (separate process/container/host)**, and the coordinator, knowing only a URL, can handle everything from
  card discovery to connection in one step with `ClientFactory.create_from_url(url)`.
- Besides JSON-RPC, the **transport bindings** include **gRPC** (`TransportProtocol.GRPC`) and **HTTP+JSON**
  (`TransportProtocol.HTTP_JSON`), advertised via the card's `supported_interfaces`.
- **Streaming**: with `AgentCapabilities(streaming=True)` + `ClientConfig(streaming=True)`, you can receive
  partial results as a `TaskStatusUpdateEvent`/`TaskArtifactUpdateEvent` stream.

### Reference: sketch of a coordinator that connects by URL only (production style)
```python
# When each agent is running as an independent service (not executed; illustrative only)
client = await ClientFactory(ClientConfig(httpx_client=hx)).create_from_url(
    "http://math-agent.internal:9001/"   # card discovery + connection in one step
)
async for resp in client.send_message(
    SendMessageRequest(message=new_text_message("3200*4", role=Role.ROLE_USER))
):
    print(get_stream_response_text(resp))
```

### Limitations / Notes
- Some models (especially small ones) may violate the decomposition (JSON) format, so a **fallback parser** is included.
- The in-memory task store (`InMemoryTaskStore`) is for demos only. Use a persistent store in production.
- Authentication/security (`SecurityScheme`), push notifications, and polling for long-running tasks are supported by the SDK but are outside the scope of this notebook.
